# PyGeoFetch — 07: Copernicus, USGS & Authenticated Providers

Deep dive into the 12 authenticated providers — Copernicus, USGS, NASA, Planet, Sentinel Hub, and more.

---
### What you'll learn
- Copernicus CDSE: full Sentinel archive (S1/S2/S3/S5P)
- USGS Earth Explorer: Landsat 1–9, ASTER, MODIS
- NASA Earthdata: MODIS, ICESat-2, GEDI, VIIRS
- NASA Earthdata Cloud: direct S3 access
- Planet Labs: PlanetScope + SkySat asset activation
- Sentinel Hub: on-the-fly processing
- OpenTopography: DEM access
- Alaska SAR Facility: Sentinel-1 + ALOS PALSAR

---
> **Note:** All cells are runnable but require valid credentials stored via `PyGeoFetch auth add`

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.core.authenticator import AuthManager
from pathlib import Path
import pandas as pd

client = PyGeoFetch(log_level="INFO")
auth = AuthManager()
Path("output/auth_providers").mkdir(parents=True, exist_ok=True)

# Check which authenticated providers are ready
stored = {e['provider']: e for e in auth.list()}
print("Stored credentials:")
AUTH_PROVIDERS = [
    "copernicus", "usgs", "nasa_earthdata", "nasa_earthdata_cloud",
    "planet", "sentinel_hub", "opentopography",
    "alaska_satellite_facility", "maxar_gbdx", "airbus_oneatlas",
    "google_earth_engine", "terrabotics"
]
for p in AUTH_PROVIDERS:
    status = "✓ configured" if p in stored else "✗ not configured"
    print(f"  {p:<35} {status}")

## 1. Copernicus CDSE — Full Sentinel Archive

In [ ]:
# Copernicus offers Sentinel-1, 2, 3, 5P — free with registration
# Register at: https://dataspace.copernicus.eu/

# Add credentials if not already stored
# client.add_credentials("copernicus", username="email@example.com", password="YOUR_PASS")

# Test authentication
try:
    session = auth.authenticate("copernicus")
    print(f"✓ Copernicus authenticated (expires {str(session.expires_at)[:19]})")
    COPERNICUS_READY = True
except Exception as e:
    print(f"✗ Copernicus not authenticated: {e}")
    print("  Add credentials: client.add_credentials('copernicus', username=..., password=...)")
    COPERNICUS_READY = False

In [ ]:
if COPERNICUS_READY:
    # Sentinel-2 L2A — atmospherically corrected, full resolution
    q_s2 = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        start_date="2024-01-01",
        end_date="2024-06-01",
        cloud_cover_max=15,
        satellites=["Sentinel-2"],
        max_results=20,
        sort_by="cloud_cover",
        sort_ascending=True,
    )

    results_s2 = client.search(q_s2, providers=["copernicus"])
    print(f"Copernicus Sentinel-2 results: {len(results_s2)}")

    for r in results_s2[:5]:
        dt = str(r.properties.get("datetime", ""))[:10] if r.properties else "?"
        size = r.properties.get("size") if r.properties else None
        size_str = f"{int(size)/1e9:.1f} GB" if size else "unknown"
        print(f"  {dt}  cloud={r.cloud_cover:.0f}%  size≈{size_str}  {r.id[:50]}")
else:
    print("[Skipped] Add Copernicus credentials to run this cell")

In [ ]:
if COPERNICUS_READY:
    # Sentinel-1 SAR — all-weather, day/night radar imagery
    q_s1 = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        start_date="2024-01-01",
        end_date="2024-03-01",
        satellites=["Sentinel-1"],
        max_results=10,
    )

    results_s1 = client.search(q_s1, providers=["copernicus"])
    print(f"Copernicus Sentinel-1 (SAR) results: {len(results_s1)}")

    for r in results_s1[:3]:
        dt = str(r.properties.get("datetime", ""))[:10] if r.properties else "?"
        mode = r.properties.get("operationalMode", "?") if r.properties else "?"
        pol  = r.properties.get("polarisationMode", "?") if r.properties else "?"
        print(f"  {dt}  mode={mode}  polarisation={pol}  {r.id[:45]}")

    # Note: Sentinel-1 has no cloud cover — it's radar!
    print("\n  Note: SAR (radar) data is cloud-independent — works in any weather")
else:
    print("[Skipped] Add Copernicus credentials to run this cell")

## 2. USGS Earth Explorer — Landsat 1–9

In [ ]:
# USGS — free with registration at ers.cr.usgs.gov
# Provides Landsat Collection 2, ASTER, MODIS, Hyperion, EO-1, SRTM

# client.add_credentials("usgs", username="YOUR_USERNAME", password="YOUR_PASSWORD")

try:
    session = auth.authenticate("usgs")
    print(f"✓ USGS authenticated")
    USGS_READY = True
except Exception as e:
    print(f"✗ USGS not authenticated: {e}")
    USGS_READY = False

if USGS_READY:
    # Search Landsat 8/9 Collection 2 Tier 1
    q_landsat = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        start_date="2024-01-01",
        end_date="2024-06-01",
        cloud_cover_max=20,
        satellites=["Landsat"],
        max_results=20,
        sort_by="cloud_cover",
        sort_ascending=True,
    )

    results_landsat = client.search(q_landsat, providers=["usgs"])
    print(f"USGS Landsat results: {len(results_landsat)}")

    for r in results_landsat[:5]:
        dt = str(r.properties.get("datetime", ""))[:10] if r.properties else "?"
        print(f"  {dt}  cloud={r.cloud_cover:.0f}%  {r.satellite}  {r.id[:50]}")

## 3. NASA Earthdata — MODIS, ICESat-2, GEDI

In [ ]:
# NASA Earthdata — free at urs.earthdata.nasa.gov
# Same account used for alaska_satellite_facility and nasa_earthdata_cloud

# client.add_credentials("nasa_earthdata", username="USER", password="PASS")

try:
    session = auth.authenticate("nasa_earthdata")
    print(f"✓ NASA Earthdata authenticated")
    NASA_READY = True
except Exception as e:
    print(f"✗ NASA Earthdata not authenticated: {e}")
    NASA_READY = False

if NASA_READY:
    # MODIS Terra daily surface reflectance
    q_modis = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        start_date="2024-01-01",
        end_date="2024-02-01",
        satellites=["Terra"],
        max_results=10,
    )

    results_modis = client.search(q_modis, providers=["nasa_earthdata"])
    print(f"NASA MODIS Terra results: {len(results_modis)}")

    for r in results_modis[:3]:
        dt = str(r.properties.get("datetime", ""))[:10] if r.properties else "?"
        print(f"  {dt}  {r.collection}  {r.id[:50]}")

## 4. Planet Labs — Daily PlanetScope 3m

In [ ]:
# Planet — subscription required but trials available
# API key from: planet.com/account

# client.add_credentials("planet", api_key="YOUR_PLANET_API_KEY")

try:
    session = auth.authenticate("planet")
    print(f"✓ Planet authenticated")
    PLANET_READY = True
except Exception as e:
    print(f"✗ Planet not authenticated: {e}")
    PLANET_READY = False

if PLANET_READY:
    # PlanetScope 3m daily imagery
    q_planet = SearchQuery(
        bbox=BoundingBox.from_string("-74.01,40.70,-73.97,40.74"),  # Small downtown Manhattan
        start_date="2024-06-01",
        end_date="2024-06-15",
        cloud_cover_max=10,
        max_results=10,
    )

    results_planet = client.search(q_planet, providers=["planet"])
    print(f"Planet PlanetScope results: {len(results_planet)}")

    for r in results_planet[:5]:
        dt = str(r.properties.get("datetime", ""))[:16] if r.properties else "?"
        item_type = r.properties.get("item_type", "?") if r.properties else "?"
        print(f"  {dt}  type={item_type}  cloud={r.cloud_cover:.0f}%  {r.id}")

    print()
    print("Note: Planet assets require activation before download.")
    print("PyGeoFetch handles this automatically — activation runs before download.")
    print("Activation typically takes 30 seconds to 5 minutes.")

## 5. OpenTopography — DEM Access

In [ ]:
# OpenTopography provides SRTM, COP-DEM, ALOS, and LiDAR
# Free API key at: portal.opentopography.org

# client.add_credentials("opentopography", api_key="YOUR_KEY")

try:
    session = auth.authenticate("opentopography")
    print(f"✓ OpenTopography authenticated")
    OT_READY = True
except Exception as e:
    print(f"✗ OpenTopography not authenticated: {e}")
    OT_READY = False

if OT_READY:
    # Available DEM datasets
    q_dem = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        max_results=10,
    )

    results_dem = client.search(q_dem, providers=["opentopography"])
    print(f"OpenTopography DEM datasets: {len(results_dem)}")

    for r in results_dem:
        res = r.properties.get("resolution", "?") if r.properties else "?"
        print(f"  {r.id:<35}  collection={r.collection}  res={res}m")

    # Download Copernicus DEM (1 arc-sec = ~30m)
    if results_dem:
        cop_dem = next((r for r in results_dem if "COP" in r.id.upper() or "copernicus" in r.id.lower()), results_dem[0])
        print(f"\nDownloading: {cop_dem.id}")
        options_dem = DownloadOptions(parallel=1, retry_attempts=3, resume=True, on_failure="skip")
        dl = client.download([cop_dem], Path("output/auth_providers/dem"), options_dem)
        for r in dl:
            if r.success:
                mb = r.bytes_downloaded / (1024*1024) if r.bytes_downloaded else 0
                print(f"  ✓ Downloaded: {mb:.1f} MB")
                for p in r.output_paths:
                    print(f"    {p}")

## 6. Alaska SAR Facility — Sentinel-1 + ALOS PALSAR

In [ ]:
# ASF DAAC — free with NASA Earthdata account
# Specialises in SAR: Sentinel-1, ALOS PALSAR, ERS-1/2, JERS-1

# client.add_credentials("alaska_satellite_facility", username="USER", password="PASS")

try:
    session = auth.authenticate("alaska_satellite_facility")
    print(f"✓ ASF authenticated")
    ASF_READY = True
except Exception as e:
    print(f"✗ ASF not authenticated: {e}")
    ASF_READY = False

if ASF_READY:
    # Sentinel-1 GRD (Ground Range Detected) — most common format
    q_sar = SearchQuery(
        bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
        start_date="2024-01-01",
        end_date="2024-03-01",
        satellites=["Sentinel-1"],
        max_results=10,
    )

    results_sar = client.search(q_sar, providers=["alaska_satellite_facility"])
    print(f"ASF Sentinel-1 results: {len(results_sar)}")

    for r in results_sar[:5]:
        dt = str(r.properties.get("datetime", ""))[:10] if r.properties else "?"
        mode     = r.properties.get("beamModeType", "?") if r.properties else "?"
        pol      = r.properties.get("polarization", "?") if r.properties else "?"
        orbit    = r.properties.get("flightDirection", "?") if r.properties else "?"
        print(f"  {dt}  mode={mode}  pol={pol}  orbit={orbit}")

    print()
    print("SAR processing levels available:")
    print("  GRD (Ground Range Detected) — amplitude, most common")
    print("  SLC (Single Look Complex)   — phase preserved, for InSAR")
    print("  RTC (Radiometric Terrain Corrected) — terrain corrected amplitude")

## 7. Provider Comparison — Same AOI, All Sources

In [ ]:
# Search all available authenticated providers simultaneously
# Uses only providers where credentials are stored

available_auth = [e['provider'] for e in auth.list()]
free_providers = ["aws_earth", "planetary_computer", "element84"]
all_available = free_providers + [p for p in available_auth if p not in free_providers]

print(f"Searching {len(all_available)} available providers: {all_available}")

q_all = SearchQuery(
    bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
    start_date="2024-03-01",
    end_date="2024-06-01",
    cloud_cover_max=20,
    max_results=10,
)

all_results = client.search(q_all, providers=all_available)
print(f"\nTotal unique results: {len(all_results)}")

from collections import Counter
by_provider = Counter(r.provider for r in all_results)
by_satellite = Counter(r.satellite for r in all_results)

print("\nResults by provider:")
for prov, count in by_provider.most_common():
    print(f"  {prov:<35} {count} scenes")

print("\nResults by satellite:")
for sat, count in by_satellite.most_common(10):
    print(f"  {str(sat):<35} {count} scenes")

---
## Summary

| Provider | Data Type | Resolution | Revisit | Cost |
|---|---|---|---|---|
| Copernicus | Sentinel-1/2/3/5P | 5–300m | 1–5 days | Free (register) |
| USGS | Landsat 1–9, ASTER | 15–100m | 16 days | Free (register) |
| NASA Earthdata | MODIS, VIIRS, ICESat-2, GEDI | 250m–1km | Daily | Free (register) |
| Planet | PlanetScope, SkySat | 0.5–3m | Daily | Subscription |
| Sentinel Hub | Sentinel + Landsat | 10–30m | 5+ days | Freemium |
| OpenTopography | SRTM, COP-DEM, LiDAR | 1–90m | Static | Free (API key) |
| ASF DAAC | Sentinel-1, ALOS PALSAR | 5–20m | 12 days | Free (register) |

**Next:** `08_cli_complete_reference.ipynb` — every CLI command with examples.